# 07 · Baseline evaluation (`mixedbread-ai/mxbai-embed-large-v1`, no fine-tuning)

Evaluates the **unmodified** mxbai model on the 394-query test set.  
Corpus embeddings were pre-computed with the same model in notebook 03 (`artifacts/embeddings_mxbai_base.npy`).

Metric: a query is a hit at rank *k* if the correct title appears in the top-*k* results (fuzzy title match, `SequenceMatcher` ratio ≥ 0.85). Per-query results are written to `results/`.

In [ ]:
import numpy as np
import pickle
import faiss
import pandas as pd
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer

## 1. Load pre-computed corpus embeddings

In [ ]:
EMBEDDINGS_PATH = "../artifacts/embeddings_mxbai_base.npy"
METADATA_PATH   = "../artifacts/metadata.pkl"

embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

faiss.normalize_L2(embeddings)
print(f"Corpus: {embeddings.shape[0]:,} movies  |  dim={embeddings.shape[1]}")

## 2. Build FAISS index

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"Index ready: {index.ntotal:,} vectors")

## 3. Load baseline query encoder

In [ ]:
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
print("Model loaded:", model)

## 4. Search helpers

In [ ]:
def embed_query(text: str) -> np.ndarray:
    prefixed = f"Represent this sentence for searching relevant passages: {text}"
    vec = model.encode([prefixed]).astype("float32")
    faiss.normalize_L2(vec)
    return vec


def search(query_text: str, top_k: int = 10) -> list[dict]:
    scores, indices = index.search(embed_query(query_text), top_k)
    return [
        {"title": metadata[idx]["title"], "similarity": float(score)}
        for score, idx in zip(scores[0], indices[0])
    ]


def fuzzy_match(a: str, b: str, threshold: float = 0.85) -> tuple[bool, float]:
    ratio = SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()
    return ratio >= threshold, ratio

## 5. Run evaluation on test set

In [ ]:
df_test = pd.read_csv("../data/eval/test_set_394.csv")
print(f"Test queries: {len(df_test)}")
df_test["query_type"].value_counts()

In [ ]:
rows = []

for _, row in df_test.iterrows():
    results = search(row["query"], top_k=10)

    correct_rank = match_ratio = correct_score = None
    for rank, r in enumerate(results, 1):
        ok, ratio = fuzzy_match(r["title"], row["title"])
        if ok:
            correct_rank, match_ratio, correct_score = rank, ratio, r["similarity"]
            break

    rows.append({
        "title":            row["title"],
        "year":             row.get("release_year"),
        "genre":            row.get("genre"),
        "query":            row["query"],
        "query_type":       row["query_type"],
        "top1_result":      results[0]["title"] if results else None,
        "correct_rank":     correct_rank if correct_rank is not None else "not found",
        "match_ratio":      round(match_ratio, 3) if match_ratio else None,
        "hit@1":            int(correct_rank == 1) if correct_rank else 0,
        "hit@5":            int(correct_rank is not None and correct_rank <= 5),
        "hit@10":           int(correct_rank is not None and correct_rank <= 10),
        "reciprocal_rank":  (1 / correct_rank) if correct_rank else 0.0,
        "similarity_score": correct_score or 0.0,
    })

results_df = pd.DataFrame(rows)
results_df.to_csv("../results/eval_394_mxbai_base.csv", index=False)
print("Saved → ../results/eval_394_mxbai_base.csv")

## 6. Results

In [ ]:
# Overall metrics
overall = {
    "Hit@1":  results_df["hit@1"].mean(),
    "Hit@5":  results_df["hit@5"].mean(),
    "Hit@10": results_df["hit@10"].mean(),
    "MRR":    results_df["reciprocal_rank"].mean(),
}
print("=== Baseline (mxbai-embed-large-v1) ===")
for k, v in overall.items():
    print(f"  {k}: {v:.1%}")

In [ ]:
# Per query-type breakdown
print("\n=== Per query type (Hit@1 / Hit@10) ===")
for qt in ["oracle", "conversational", "naturalistic", "vague"]:
    sub = results_df[results_df["query_type"] == qt]
    if len(sub):
        print(f"  {qt:>15s}: {sub['hit@1'].mean():.1%} / {sub['hit@10'].mean():.1%}  (n={len(sub)})")

In [ ]:
# Summary table
summary = results_df.groupby("query_type").agg(
    n=("hit@1", "count"),
    Hit_at_1=("hit@1", "mean"),
    Hit_at_5=("hit@5", "mean"),
    Hit_at_10=("hit@10", "mean"),
    MRR=("reciprocal_rank", "mean"),
).reset_index()
summary[["Hit_at_1", "Hit_at_5", "Hit_at_10", "MRR"]] = summary[
    ["Hit_at_1", "Hit_at_5", "Hit_at_10", "MRR"]
].applymap(lambda x: f"{x:.1%}")
display(summary)